# Controlled Colab V2 — setup validation only

This notebook validates the locked, development-only baseline environment. It contains no training or official-test command. Any failed gate raises immediately; do not bypass a failure. Provision the repository checkout at `/content/SS-VIRULEX-COVID-CT` before starting and select a Colab GPU runtime at the outset.

Colab's unrelated preinstalled packages can make global `pip check` report conflicts. That output is preserved as diagnostic evidence, while the hard gate requires every project distribution and imported module to match its exact intended version. Competing OpenCV wheels are removed before installation; validation rejects any remaining distribution that shares the `cv2` namespace and requires `cv2.__version__ == 4.10.0`.

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')
REPO = Path('/content/SS-VIRULEX-COVID-CT')
PACKAGE = Path('/content/drive/MyDrive/colab_package')
OUTPUT_ROOT = Path('/content/drive/MyDrive/SS_VIRULEX_Reliability_V2')
COMPLETED_ROOT = Path('/content/drive/MyDrive/SS_VIRULEX_Reliability')

assert REPO.is_dir(), f'Missing repository checkout: {REPO}'
assert (REPO / '.git').exists(), f'Not a Git checkout: {REPO}'
assert PACKAGE.is_dir(), f'Missing preserved package: {PACKAGE}'
assert OUTPUT_ROOT != COMPLETED_ROOT
assert PACKAGE not in OUTPUT_ROOT.parents and OUTPUT_ROOT not in PACKAGE.parents
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
PATH_GATE_PASSED = True
print('Authoritative repository:', REPO)
print('Read-only package input:', PACKAGE)
print('New V2 output root:', OUTPUT_ROOT)

In [ ]:
import subprocess

assert PATH_GATE_PASSED
VALIDATED_CHECKPOINT = '78a06737d67b46b3961dab2f8a26c16db7de2b24'
PROTECTED_PATHS = [
    '04_reliability/configs/encoder_gradual_final_block.yaml',
    '04_reliability/ss_virulex_reliability/audit_leakage.py',
    '04_reliability/ss_virulex_reliability/common.py',
    '04_reliability/ss_virulex_reliability/encoder_experiments.py',
    '04_reliability/ss_virulex_reliability/fold_safe_fusion.py',
    '04_reliability/ss_virulex_reliability/med_data.py',
    '04_reliability/ss_virulex_reliability/oof_med_micn.py',
    '04_reliability/ss_virulex_reliability/trainers.py',
    'results/reliability_colab/final_test/provenance/LOCKED_ENCODER_CONFIG.yaml',
]

def git(*args, check=True):
    return subprocess.run(
        ['git', '-C', str(REPO), *args], check=check,
        capture_output=True, text=True
    )

branch = git('branch', '--show-current').stdout.strip()
head = git('rev-parse', 'HEAD').stdout.strip()
assert branch == 'reliability-integration', f'Unexpected branch: {branch}'
assert git('rev-parse', '--verify', f'{VALIDATED_CHECKPOINT}^{{commit}}').stdout.strip() == VALIDATED_CHECKPOINT
assert git('merge-base', '--is-ancestor', VALIDATED_CHECKPOINT, 'HEAD', check=False).returncode == 0, (
    f'Validated checkpoint is not an ancestor of HEAD {head}'
)
status = git('status', '--porcelain').stdout.strip()
assert not status, f'Tracked Git changes are present:\n{status}'
protected_diff = git(
    'diff', '--exit-code', VALIDATED_CHECKPOINT, 'HEAD', '--', *PROTECTED_PATHS, check=False
)
assert protected_diff.returncode == 0, 'Validated implementation/config files changed after the checkpoint'
REPOSITORY_GATE_PASSED = True
print('Branch:', branch)
print('HEAD:', head)
print('Validated ancestor:', VALIDATED_CHECKPOINT)
print('Tracked status: clean')

In [ ]:
import os
import sys

assert REPOSITORY_GATE_PASSED
requirements = PACKAGE / 'requirements_colab.txt'
setup_tools = REPO / '04_reliability/configs/v2/setup_tools.txt'
assert requirements.is_file(), f'Missing requirements: {requirements}'
assert setup_tools.is_file(), f'Missing setup-tool pins: {setup_tools}'
opencv_namespace_distributions = [
    'opencv-python', 'opencv-contrib-python',
    'opencv-contrib-python-headless', 'opencv-python-headless',
]
subprocess.run(
    [sys.executable, '-m', 'pip', 'uninstall', '-y', *opencv_namespace_distributions],
    check=True,
)
subprocess.run(
    [
        sys.executable, '-m', 'pip', 'install', '-q',
        '-r', str(requirements), '-r', str(setup_tools),
    ],
    check=True,
)
test_environment = os.environ.copy()
test_environment['PYTHONPATH'] = str(REPO / '04_reliability')
subprocess.run(
    [sys.executable, '-m', 'pytest', '04_reliability/tests', '-q'],
    cwd=REPO, env=test_environment, check=True,
)
TEST_GATE_PASSED = True
print('Dependency installation completed and lightweight tests passed.')

In [ ]:
from datetime import datetime, timezone

assert TEST_GATE_PASSED
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
RUN_OUTPUT = OUTPUT_ROOT / '00_setup_validation' / RUN_ID
assert not RUN_OUTPUT.exists(), f'Refusing to overwrite: {RUN_OUTPUT}'
validation_environment = os.environ.copy()
validation_environment['PYTHONPATH'] = str(REPO / '04_reliability')
subprocess.run(
    [
        sys.executable, '-m', 'ss_virulex_reliability.validate_v2_setup',
        '--repo-root', str(REPO),
        '--package-root', str(PACKAGE),
        '--output-root', str(OUTPUT_ROOT),
        '--run-id', RUN_ID,
    ],
    cwd=REPO, env=validation_environment, check=True,
)
VALIDATION_REPORT = RUN_OUTPUT / 'SETUP_VALIDATION_COMPLETE.json'
assert VALIDATION_REPORT.is_file(), f'Missing completion report: {VALIDATION_REPORT}'
VALIDATION_GATE_PASSED = True

In [ ]:
import json

assert VALIDATION_GATE_PASSED
report = json.loads(VALIDATION_REPORT.read_text(encoding='utf-8'))
assert report['status'] == 'PASS'
assert report['training_performed'] is False
assert report['official_test_rows_accessed'] == 0
assert report['development_contract']['rows'] == 1054
assert report['development_contract']['concept_count'] == 8
assert report['development_contract']['group_overlap_count'] == 0
assert report['locked_encoder']['configuration_id'] == '112592618942'
SETUP_READY_FOR_SEPARATELY_AUTHORIZED_BASELINE = True
print(json.dumps({
    'status': report['status'],
    'head': report['repository']['head'],
    'cuda_device': report['dependencies']['cuda_device'],
    'development_rows': report['development_contract']['rows'],
    'splits': report['development_contract']['split_counts'],
    'concept_count': report['development_contract']['concept_count'],
    'group_overlap_count': report['development_contract']['group_overlap_count'],
    'leakage_audit_status': report['leakage_audit']['overall_status'],
    'configuration_id': report['locked_encoder']['configuration_id'],
    'report': str(VALIDATION_REPORT),
}, indent=2))
print('STOP: setup gates passed. This notebook performs no training and no official-test evaluation.')